# Step 03. Hybrid A* 알고리즘

> **출처**: PythonRobotics (https://github.com/AtsushiSakai/PythonRobotics)  
> Original author: Atsushi Sakai (@Atsushi_twi) / License: MIT  
>  
> **교육 주관**: Hanyang University, Automotive Intelligence Laboratory (AILAB)

A\*는 차의 **상태**(탐색이 기억하는 위치·방향)를 격자 칸 (ix,iy)로만 표현해 경로가 꺾입니다.  
Hybrid A\*는 상태를 연속 **(x, y, yaw)** 로 확장하고, 이웃 확장에 **Bicycle Model**을 써서 차량이 실제로 주행 가능한 부드러운 경로를 찾습니다.

> 작은 조각으로 나눠 이해하고, 마지막에 **격자 경로 vs Hybrid 경로**를 비교합니다.

## 학습 목표

1. A\* vs Hybrid A\* 상태 공간 차이: (ix,iy) -> (x,y,yaw)
2. Bicycle Model 운동 방정식과 최소 회전반경 (섹션 2, TODO 1: yaw 갱신)
3. 비슷한 상태를 같은 칸(버킷) (ix,iy,iyaw)으로 묶어 중복 탐색 막기 (섹션 4)
4. 조향 후보로 이웃 만들기와 비용 설계 (섹션 5, TODO 2: 이웃 비용)
5. 격자 꺾임 경로 vs Hybrid 부드러운 경로를 수치로 비교 (섹션 8)

In [ ]:
%matplotlib inline
import sys, os, math, heapq, time
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../utils/01_global_planning'))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

from grid_map import MakeSimpleMap
from vehicle_model import PlotCar, WB

ox, oy, sx, sy, gx, gy = MakeSimpleMap()   # o*: 장애물(obstacle) 좌표, s*: 출발점(start), g*: 목표점(goal)
print(f'환경 로드 완료 — 차량 축간거리 WB = {WB} m')
print(f'출발 ({sx},{sy})  ->  목표 ({gx},{gy})')


## 1. A\*와 Hybrid A\*의 차이

| 구분 | A\* | Hybrid A\* |
|------|------|----------|
| 상태 | 격자 (ix,iy) | 연속 (x,y,yaw) |
| 이동 | 8방향 격자 | Bicycle Model |
| 키 | (ix,iy) | (ix,iy,iyaw) |
| 경로 | 꺾임 | 부드러움(주행 가능) |

Heuristic은 동일(Euclidean). **이웃 확장 방식**만 다릅니다.

## 2. Bicycle Model 운동 방정식

$$x_{new} = x + ds\cos\theta,\quad y_{new} = y + ds\sin\theta,\quad \theta_{new} = \theta + \frac{ds\,\tan\delta}{WB}$$

| 기호 | 의미 | 코드 |
|---|---|---|
| $ds$ | 한 스텝 이동거리 (2 m) | `step_size` |
| $\delta$ | 조향각 (앞바퀴 각도, rad) | `steer` |
| $WB$ | 축간거리(휠베이스, 앞뒤 바퀴축 사이, 3.0 m) | `wheelbase` (= `WB`) |
| $\theta$ | 진행 방향 yaw (rad) | `yaw` |

- 같은 조향각을 유지하면 반지름 $R = WB / \tan\delta$ 인 원을 돎 -> 최대 조향 40°에서 **최소 회전반경** $R_{min} = 3.0 / \tan 40° \approx 3.6$ m
- 격자 A\*는 한 점에서 방향을 90°까지 바로 바꾸지만(회전반경 0), Hybrid A\*는 $R_{min}$ 보다 급하게 꺾을 수 없음 -> 차가 실제로 따라갈 수 있는 경로
- 계산 순서: 먼저 현재 yaw 로 $ds$ 전진 -> yaw 갱신 (오일러 적분). 그래서 1스텝만으로는 y 가 변하지 않음 (아래 표는 3스텝)

### 실습 (TODO 1)

아래 `BicycleStep` 의 yaw 갱신 한 줄을 위 수식대로 채웁니다.

In [ ]:
def BicycleStep(x, y, yaw, steer, step_size=2.0, wheelbase=WB):
    # 전륜 조향 Bicycle Model 한 스텝: 현재 yaw 로 step_size 전진 -> yaw 갱신
    x   += step_size * math.cos(yaw)
    y   += step_size * math.sin(yaw)

    ##### TODO 1 #####
    # yaw 갱신하기: θ_new = θ + ds · tanδ / WB   (위 수식, ds = step_size, δ = steer, WB = wheelbase)
    # 사용함수: math.tan(rad) / 기능: 탄젠트 (steer 는 rad)
    # 주의: tan 안에는 steer 만 -> tan(steer / wheelbase) 는 오답
    #################

    yaw = ...

    #################
    yaw  = (yaw + math.pi) % (2*math.pi) - math.pi   # -pi~pi 유지
    return x, y, yaw

# 한 스텝은 '현재 yaw 로 전진한 뒤 yaw 갱신' 이라 1스텝만으로는 y 가 변하지 않는다 -> 3스텝(6m) 유지해 비교
print('조향각별 3스텝 결과 (출발 원점/직진, step=2m x 3)')
print(f'  {"steer":>7}  {"x":>6}  {"y":>6}  {"yaw_new":>9}')
for d in [-40, -20, 0, 20, 40]:
    nx, ny, nyaw = 0.0, 0.0, 0.0   # nx, ny, nyaw: 스텝 진행 후 상태 (next x, y, yaw)
    for _ in range(3):
        nx, ny, nyaw = BicycleStep(nx, ny, nyaw, math.radians(d))
    print(f'  {d:>6}deg  {nx:>6.2f}  {ny:>6.2f}  {math.degrees(nyaw):>7.1f}deg')

R_MIN = WB / math.tan(math.radians(40))   # 최소 회전반경 (섹션 8 에서도 사용)

# 확인: 20deg 1스텝 -> yaw 변화 = 2·tan(20°)/3.0 rad (= 13.9deg)
_, _, _yaw = BicycleStep(0.0, 0.0, 0.0, math.radians(20))
assert abs(math.degrees(_yaw) - 13.90) < 0.01, f'TODO 1 확인 - yaw 갱신 오답: 20deg 1스텝 기대값 13.9deg (현재 {math.degrees(_yaw):.2f}deg)'
print(f'✅ BicycleStep 확인 통과 | 최소 회전반경 R_min = WB / tan(40deg) = {R_MIN:.2f} m')

## 3. 조향각별 궤적

한 조향각을 계속 유지하면 원호(반지름 $WB/\tan\delta$)가 됩니다. 5가지 조향각(탐색에 쓰는 후보와 같은 -40, -20, 0, +20, +40도)을 각각 15스텝(30 m) 유지하며 한 스텝씩 그려 비교합니다.

In [ ]:
# 각 조향각을 계속 유지하면 곡선 궤적이 됨 (한 스텝씩 그려가며 확인)
STEERS = [-40, -20, 0, 20, 40]   # 섹션 5 의 조향 후보와 같음
COLORS = ['#d62728','#ff7f0e','#2ca02c','#1f77b4','#9467bd']
N = 15
trajs = []
for d in STEERS:
    x, y, yaw = 0.0, 0.0, 0.0; xs, ys = [0.0], [0.0]
    for _ in range(N):
        x, y, yaw = BicycleStep(x, y, yaw, math.radians(d)); xs.append(x); ys.append(y)
    trajs.append((xs, ys))

fig, ax = plt.subplots(figsize=(8, 6))
ax.set_xlim(-2, 33); ax.set_ylim(-20, 20); ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.plot(0, 0, 'og', ms=10); ax.set_title('Bicycle Model: constant-steering trajectories')
lines = [ax.plot([], [], '-', color=c, lw=2, label=f'{d}deg')[0] for c, d in zip(COLORS, STEERS)]
ax.legend(loc='upper left', fontsize=8)

def Upd(f):
    for ln, (xs, ys) in zip(lines, trajs): ln.set_data(xs[:f+2], ys[:f+2])
    return lines
HTML(FuncAnimation(fig, Upd, frames=N, interval=200, repeat=False, blit=False).to_jshtml())


## 4. HybridNode 클래스

연속 상태 (x,y,yaw)를 저장하되, 중복 탐색 방지를 위해 **이산 버킷 인덱스 (ix,iy,iyaw)** 도 함께 가집니다.  
같은 버킷이면 이미 방문한 상태로 취급합니다.

In [ ]:
RESOLUTION = 2.0
YAW_RES    = math.radians(15.0)   # yaw를 15도 단위로 이산화 (버킷 24개)
N_YAW      = round(2 * math.pi / YAW_RES)   # 24

class HybridNode:
    # 연속 상태 (x,y,yaw) + 이산 버킷 인덱스 (ix,iy,iyaw)
    def __init__(self, x, y, yaw, cost, parent_index, steer=0.0):
        self.x, self.y, self.yaw = x, y, yaw
        self.steer = steer                 # 이 상태로 올 때 쓴 조향각 (조향 변화 비용 계산용)
        self.cost = cost
        self.parent_index = parent_index
        self.ix = round(x / RESOLUTION)
        self.iy = round(y / RESOLUTION)
        self.iyaw = round(yaw / YAW_RES) % N_YAW   # -180도와 +180도는 같은 방향 -> 같은 버킷 (나머지 연산으로 0~23)
    def index(self):
        return (self.ix, self.iy, self.iyaw)   # dict 키 (중복 제거용)
    def __lt__(self, other):
        return self.cost < other.cost

hn = HybridNode(5.0, 3.0, math.radians(30), 2.5, None)
print(f'예시: x={hn.x}, y={hn.y}, yaw=30deg -> 버킷 키 index()={hn.index()}  (x/2=2.5, y/2=1.5 -> round 는 짝수 쪽: 2, 2 / Step 01 섹션 5)')
print('A*는 키가 (ix,iy) 2D, Hybrid A*는 (ix,iy,iyaw) 3D (방향까지 구분)')


## 5. 이웃 확장 (조향 후보)

A\*의 8방향 대신, 조향각 -40~+40도(20도 간격: -40, -20, 0, +20, +40도)에 Bicycle Model을 적용해 이웃 5개를 만듭니다.

이웃의 누적 비용 $g$ = 부모 비용 + 이동 거리 + 페널티 2개:

| 항 | 식 | 역할 |
|---|---|---|
| 이동 거리 | `STEP_SIZE` (2 m) | 짧은 경로 선호 |
| 급조향 페널티 | `|steer| × STEER_COST` (0.1) | 직진 선호. steer 는 rad 이라 40도 = 0.70 rad -> 0.07 (작은 영향) |
| 조향 변화 페널티 | `|steer − 부모 steer| × STEER_CHANGE_COST` (2.0) | 좌우로 번갈아 꺾는 지그재그 억제 (주된 부드러움 요인) |

### 실습 (TODO 2)

`GetNeighbors` 의 이웃 비용 한 줄을 위 표대로 채웁니다.

In [ ]:
STEER_RES = math.radians(20.0); MAX_STEER = math.radians(40.0); STEP_SIZE = 2.0
STEER_COST        = 0.1   # |δ| 에 곱하는 비용 (rad 기준 -> 영향 작음)
STEER_CHANGE_COST = 2.0   # 직전 조향과의 차이 |Δδ| 에 곱하는 비용 (클수록 부드럽지만 탐색이 굼뜸)

def GetNeighbors(node):
    # 조향각 -40~+40도(20도 간격, 직진 0도 포함)로 다음 상태 생성
    neighbors = []
    steer = -MAX_STEER
    while steer <= MAX_STEER + 1e-6:
        nx, ny, nyaw = BicycleStep(node.x, node.y, node.yaw, steer, STEP_SIZE)

        ##### TODO 2 #####
        # 이웃의 누적 비용 g 계산하기: 부모 비용 + 이동 거리 + 페널티 2개 (위 표)
        #   - 부모 비용       : node.cost
        #   - 이동 거리       : STEP_SIZE
        #   - 급조향 페널티   : |steer| · STEER_COST
        #   - 조향 변화 페널티: |steer - node.steer| · STEER_CHANGE_COST
        # 사용함수: abs(x) / 기능: 절댓값
        #################

        cost = ...

        #################
        neighbors.append(HybridNode(nx, ny, nyaw, cost, node.index(), steer))
        steer += STEER_RES
    return neighbors

nbs = GetNeighbors(HybridNode(0, 0, 0, 0, None))   # nbs: 이웃 후보 목록 (neighbors)
print(f'출발점에서 가능한 다음 상태: {len(nbs)}개 (조향각 후보 수)')

# 확인 1: 직진 출발(부모 steer = 0) -> 0deg 2.00 / ±20deg 2.73 / ±40deg 3.47
# 확인 2: 부모 steer = +20deg -> 같은 조향(+20deg)을 유지하면 조향 변화 페널티 0 -> 가장 쌈
for parent_deg in (0, 20):
    _parent = HybridNode(0, 0, 0, 0, None, steer=math.radians(parent_deg))
    _costs = [nb.cost for nb in GetNeighbors(_parent)]
    _expected = [STEP_SIZE + abs(math.radians(d)) * STEER_COST + abs(math.radians(d - parent_deg)) * STEER_CHANGE_COST
                 for d in (-40, -20, 0, 20, 40)]
    assert all(abs(c - e) < 1e-9 for c, e in zip(_costs, _expected)), \
        f'TODO 2 확인 - 이웃 비용 오답 (부모 steer {parent_deg:+d}deg): node.cost + STEP_SIZE + |steer|·STEER_COST + |steer - node.steer|·STEER_CHANGE_COST'
    print(f'✅ 이웃 비용 확인 (부모 steer {parent_deg:+d}deg): ' + ' / '.join(f'{d:+d}deg {c:.2f}' for d, c in zip((-40, -20, 0, 20, 40), _costs)))


In [ ]:
# 조향 후보를 여러 단계 조합하면 도달 가능 영역이 부채꼴로 퍼진다 (A*의 8격자 이동과 대조)
steers = list(np.arange(-MAX_STEER, MAX_STEER+1e-6, STEER_RES))
fig, ax = plt.subplots(figsize=(8, 6))

def Expand(state, depth, max_depth):
    x, y, yaw = state
    if depth >= max_depth: return
    for steer in steers:
        nx, ny, nyaw = BicycleStep(x, y, yaw, steer, STEP_SIZE)
        ax.plot([x, nx], [y, ny], '-', color='steelblue', lw=0.8, alpha=max(0.7-0.18*depth, 0.15))
        ax.plot(nx, ny, '.', color='navy', ms=3, alpha=0.5)
        Expand((nx, ny, nyaw), depth+1, max_depth)

Expand((0.0, 0.0, 0.0), 0, 4)
ax.plot(0, 0, 'og', ms=12, zorder=5, label='Start')
ax.set_aspect('equal'); ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
ax.set_title('Reachable Tree (4 steps deep) — only car-feasible curves')
plt.tight_layout(); plt.show()
print('차량 기구학을 만족하는 곡선만 생성 -> 실제 주행 가능한 경로')
print('첫 마디가 한 줄인 이유: 한 스텝은 현재 yaw 로 먼저 전진 -> 첫 스텝 이웃 5개는 위치가 같고 yaw 만 다름')


## 6. HybridAstar 구현

A\* 골격 그대로, dict 키를 `node.index()`로, 이웃 확장을 `GetNeighbors()`로 바꿉니다.  
연속 좌표라 목표 점에 정확히 닿기 어려우므로 **목표 반경 3 m**(`STEP_SIZE × 1.5`) 안에 들어오면 종료합니다 → 경로 끝이 목표에서 **최대 3 m** 떨어질 수 있음 (기본 실행 결과: 목표 1.5 m 앞에서 종료).

In [ ]:
def HeuristicH(node, gx, gy):
    return math.hypot(node.x - gx, node.y - gy)   # 목표까지 유클리드 거리

def IsValidCont(x, y, ox, oy, robot_radius=1.5):
    # 연속 좌표가 모든 장애물에서 robot_radius 이상 떨어졌는지
    return all(math.hypot(iox-x, ioy-y) > robot_radius for iox, ioy in zip(ox, oy))

print('HeuristicH / IsValidCont 정의 완료')


In [ ]:
def HybridAstar(sx, sy, syaw, gx, gy, ox, oy, robot_radius=1.5):
    start = HybridNode(sx, sy, syaw, 0.0, None)
    open_set = {start.index(): start}; closed_set = {}
    pq = [(HeuristicH(start, gx, gy), start)]
    best, best_h = start, float('inf')

    for _ in range(20000):
        if not pq: break
        _, current = heapq.heappop(pq)
        if current.index() not in open_set: continue
        open_set.pop(current.index()); closed_set[current.index()] = current
        h = HeuristicH(current, gx, gy)
        if h < best_h: best_h, best = h, current
        if h < STEP_SIZE * 1.5: break                 # 목표 반경 내 도달
        for nb in GetNeighbors(current):             # Bicycle Model 확장
            if not IsValidCont(nb.x, nb.y, ox, oy, robot_radius): continue
            if nb.index() in closed_set: continue
            if nb.index() not in open_set or open_set[nb.index()].cost > nb.cost:
                open_set[nb.index()] = nb
                heapq.heappush(pq, (nb.cost + HeuristicH(nb, gx, gy), nb))   # f = g + h

    path = []; node = best                            # 역추적 (parent 객체 체인)
    while node is not None:
        path.append(node)
        node = closed_set.get(node.parent_index) if node.parent_index is not None else None
    path.reverse()
    return path, closed_set

print('HybridAstar() 정의 완료 — A* 골격에 이웃 확장만 Bicycle Model로 교체')


## 7. 탐색 실행

출발 자세 45도로 탐색합니다. 탐색 노드 수는 A\*(735개)보다 **훨씬 많습니다** (아래 출력에서 배수 확인).
- 상태가 (ix,iy,iyaw) 3D 버킷 -> 같은 칸도 방향(24개)마다 따로 확장
- h(직선거리)가 벽을 모름 -> 벽 앞에서 막힌 방향으로 많이 확장
- 탐색 상한 20,000회 (`range(20000)`) 안에 끝나지 않으면 목표에 가장 가까운 노드까지의 경로 반환

In [ ]:
print('Hybrid A* 탐색 중...')
t0 = time.time()
path, closed_set = HybridAstar(sx, sy, math.radians(45), gx, gy, ox, oy)
dt = (time.time()-t0)*1000
path_x = [n.x for n in path]; path_y = [n.y for n in path]; path_yaw = [n.yaw for n in path]
print(f'탐색 노드 {len(closed_set):,}개,  경로 {len(path)}개 점,  {dt:.0f} ms')
print(f'목표 도달 거리 {math.hypot(path_x[-1]-gx, path_y[-1]-gy):.2f} m  (목표 반경 {STEP_SIZE * 1.5:.0f} m 안이면 종료)')
print(f'A* (Euclidean, 735개) 대비 탐색 노드 약 {len(closed_set) / 735:.0f}배')


In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(ox, oy, '.k', ms=2, label='Obstacle')
ax.plot(sx, sy, 'og', ms=10, label='Start'); ax.plot(gx, gy, 'xb', ms=10, mew=2, label='Goal')
ax.plot([n.x for n in closed_set.values()], [n.y for n in closed_set.values()],
        '.', color='cyan', ms=2, alpha=0.4, label=f'Explored ({len(closed_set):,})')
ax.plot(path_x, path_y, '-r', lw=2, label='Path')
for i in range(0, len(path), max(1, len(path)//8)):
    PlotCar(path_x[i], path_y[i], path_yaw[i], ax=ax, color='royalblue', alpha=0.5)
ax.set_aspect('equal'); ax.grid(True, alpha=0.3); ax.legend(loc='upper left', fontsize=8)
ax.set_title('Hybrid A* — (x,y,yaw) Path with Vehicle Poses')
plt.tight_layout(); plt.show()


## 8. 격자 경로 vs Hybrid 경로 비교

같은 출발/목표에서 격자 8방향 경로(왼쪽, `grid_search.DijkstraSearch` = Step 01 과 같은 Dijkstra)와 Hybrid 경로(오른쪽)를 비교합니다.
- 격자 경로: 한 점에서 방향이 45°·90° 로 바로 바뀜 (회전반경 0) -> 차량이 그대로 못 따라감
- Hybrid 경로: 스텝당 방향 변화가 조향 한계 안 -> 회전반경 $R_{min}$(3.6 m) 이상, 차가 따라갈 수 있는 경로
아래 출력에서 두 경로의 '한 점에서 최대 방향 변화'와 '최소 회전반경'을 수치로 비교합니다.

In [ ]:
# A*(격자 8방향) 경로 vs Hybrid A*(연속) 경로 비교
from grid_search import DijkstraSearch
class GridNode:
    def __init__(s, x, y, c, p): s.x, s.y, s.cost, s.parent_index = x, y, c, p
    def __lt__(s, o): return s.cost < o.cost
gpx, gpy, *_ = DijkstraSearch(sx, sy, gx, gy, ox, oy, 2.0, 1.0, GridNode)

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
for ax in axes:
    ax.plot(ox, oy, '.k', ms=2); ax.plot(sx, sy, 'og', ms=10); ax.plot(gx, gy, 'xb', ms=10, mew=2)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
axes[0].plot(gpx, gpy, '-o', color='purple', ms=3, lw=1.5, label='Grid path (8-dir)')
axes[0].set_title('A* / Dijkstra — jagged grid path'); axes[0].legend(fontsize=8)
axes[1].plot(path_x, path_y, '-r', lw=2, label='Hybrid path')
for i in range(0, len(path), max(1, len(path)//8)):
    PlotCar(path_x[i], path_y[i], path_yaw[i], ax=axes[1], color='royalblue', alpha=0.5)
axes[1].set_title('Hybrid A* — smooth, car-feasible'); axes[1].legend(fontsize=8)
plt.suptitle('Grid path (corners) vs Hybrid A* (smooth, yaw-aware)', fontsize=13)
plt.tight_layout(); plt.show()
# 수치 비교: 한 점에서 최대 방향 변화, Hybrid 는 회전반경 = 스텝 거리 / 방향 변화(rad)
def Headings(px, py):
    return np.unwrap(np.arctan2(np.diff(py), np.diff(px)))
grid_turn   = np.degrees(np.abs(np.diff(Headings(gpx, gpy)))).max()
hybrid_dyaw = np.abs(np.diff(np.unwrap(path_yaw)))
hybrid_turn = np.degrees(hybrid_dyaw).max()
hybrid_rmin = STEP_SIZE / max(hybrid_dyaw.max(), 1e-9)
print(f'격자 경로 : 한 점에서 최대 방향 변화 {grid_turn:.0f}deg (회전반경 0) -> 차량이 그대로 못 따라감')
print(f'Hybrid 경로: 스텝당 최대 방향 변화 {hybrid_turn:.0f}deg -> 최소 회전반경 약 {hybrid_rmin:.1f} m '
      f'(R_min {R_MIN:.1f} m 이상 -> 주행 가능)')
print('Day 1 완료! 다음 Day 2 — Global Path를 주행 가능한 Trajectory로 변환')
